# Week 9 Assignment — An Emergency Department

## The task

Build a **discrete event simulation of an emergency department**. Patients
arrive at random times with a random severity (1, 2 or 3). A limited number
of doctors see them, most severe first and, within a severity, in the order
they arrived. The waiting room holds a limited number of patients. You will
measure how long patients of each severity wait, how full the waiting room
gets, and how both change as you change the number of doctors.

The waiting room is a **minimum heap**, a priority queue stored in a list.
You do not write it this week: it is given below, complete and explained line
by line. Read it before you use it. The notes in `heaps.ipynb` cover the ideas
behind it.

## How this notebook works

- This notebook is the **handout**: the explanation, the heap, the scaffold and
  the tests. You do not turn it in. You write your solution in Python files
  (see *What to submit* at the end), starting by copying the cells below into
  them. Use the notebook, or any editor, to try things out.
- Code cells marked **Given** are complete. Copy them into your files
  unchanged (the heap, the tests and the report functions).
- Code cells marked **TODO** are yours to complete.
- The tests are plain `assert` functions. Each test prints `ok` or `FAIL` with
  a reason. Re-run the test cell after every change. The simulation tests fail
  on the untouched scaffold; that is expected.
- Run the notebook top to bottom. If you change a cell above, re-run
  everything below it.

As always, every function and method has **exactly one `return`**: set a
result variable to the "didn't work" value first, and overwrite it only once
the guard condition passes.

In [ ]:
# Given -- the test runner used at the end of each part.
def run_tests(prefix: str) -> None:
    """Run every function in this notebook whose name starts with `prefix`."""
    tests = [(name, fn) for name, fn in sorted(globals().items())
             if name.startswith(prefix) and callable(fn)]
    passed = 0
    for name, fn in tests:
        try:
            fn()
            print(f"ok    {name}")
            passed += 1
        except Exception as err:   # AssertionError, TypeError, IndexError...
            print(f"FAIL  {name}: {type(err).__name__}: {err}")
    print(f"\n{passed} of {len(tests)} checks passed")

---

## The waiting room: `MinHeap363`

### What it does

A priority queue is a line where the most important value gets served first,
whatever order the values arrived in. Ours always serves the **smallest**
value first. That is why a patient with severity `1` comes out before a
patient with severity `3`.

| Method | Success | Failure (nothing changes) |
|---|---|---|
| `add(value)` | places `value` in the heap | — (`add` cannot fail) |
| `get()` | removes and returns the smallest value | heap is empty → returns `None` |
| `peek()` | returns the smallest value **without** removing it | heap is empty → returns `None` |
| `size()` | returns the number of values in the heap | — |

### How it is stored

Picture a pyramid of boxes with the smallest number on top. Every box is
smaller than, or equal to, the two boxes right under it. That rule is the
**heap property**. We keep the pyramid in an ordinary Python list, level by
level, left to right, the root in `storage[0]`:

```python
storage = [1, 2, 4, 8]       # 1 on top; 2 and 4 under it; 8 under the 2
```

The parent and children of a position come from arithmetic, not pointers:

\begin{align}
\text{left}(p) & = 2p+1 \\
\text{right}(p) & = 2p+2 \\
\text{parent}(c) & = \left\lfloor \frac{c-1}{2} \right\rfloor
\end{align}

```python
(2 * 0 + 1, 2 * 0 + 2)       # children of position 0 (the 1): positions 1 and 2
(3 - 1) // 2                 # parent of position 3 (the 8): position 1 (the 2)
```

### How it keeps the rule

Adding and removing can break the heap property in exactly one place each, and
each has a repair that follows one value along one path.

- **`add` and `_bubble_up`.** `add` puts the new value at the end of the list,
  the next free spot at the bottom of the pyramid. If it is smaller than its
  parent, it is in the wrong place, so we swap them, and repeat from the new
  spot. It floats up like a cork:

  ```python
  while position > 0 and storage[position] < storage[parent]:
      swap them, then continue from the parent's position
  ```

- **`get` and `_sink_down`.** The answer is the root. We take it out, then move
  the **last** value of the list to the root, so the list stays packed with
  no gaps. That value is probably too big for the top. We compare it with its
  **smaller** child; if it is greater than that child we swap, and repeat from
  the child's position. It sinks like a stone until it is smaller than its
  children or reaches a leaf (a position with no children):

  ```python
  while the value is not at a leaf:
      pick the smaller child
      if the value is greater than that child: swap, then continue from the child
      otherwise: stop
  ```

Each repair moves along one root-to-leaf path, so each costs at most the height
of the tree, about $\log_2 n$ steps.

### Values that are pairs

The heap compares values only with `<` and `>`, so it works for anything Python
can compare, including pairs like `(2, 14)`. Python compares pairs element by
element: the first elements decide, and only on a tie does the second element
count. The simulation uses this below.

In [ ]:
# Given -- a complete minimum heap. Copy it into heap363.py unchanged.
class MinHeap363:
    """A priority queue that always returns its smallest value, stored as a
    complete binary tree in a Python list.

    storage -- the tree, level by level, left to right; the root is storage[0]
    """

    def __init__(self):
        # The list starts empty: a new heap holds nothing.
        self.storage: list = []

    # ---- index arithmetic and swapping

    def _left_child(self, parent: int) -> int:
        """Return the position of the left child of the node at `parent`."""
        # Each level doubles, so the children of position p sit at 2p+1 and 2p+2.
        return 2 * parent + 1

    def _right_child(self, parent: int) -> int:
        """Return the position of the right child of the node at `parent`."""
        # The right child is always right next to the left child.
        return 2 * parent + 2

    def _parent(self, child: int) -> int:
        """Return the position of the parent of the node at `child`."""
        # The inverse of the two formulas above; // throws away the remainder,
        # so both children of a node land on the same parent.
        return (child - 1) // 2

    def _swap(self, i: int, j: int) -> None:
        """Exchange the values at positions i and j."""
        # Save one value first; otherwise the first assignment would
        # overwrite it and we would lose it.
        temp = self.storage[i]
        # Copy the value at j into position i.
        self.storage[i] = self.storage[j]
        # Put the saved value into position j.
        self.storage[j] = temp

    def _is_leaf(self, position: int) -> bool:
        """Return True if position has no children in storage. A position has
        a child exactly when its left child is inside the list."""
        # The left child is the first child to appear in the list, so if even
        # that one is past the end, the node has no children at all.
        leaf = self._left_child(position) >= len(self.storage)
        return leaf

    # ---- restoring the heap property

    def _bubble_up(self) -> None:
        """Move the last value in storage up until its parent is not greater
        than it, or it reaches the root."""
        # The value that was just added is the last one in the list.
        position = len(self.storage) - 1
        # Keep climbing while we are not at the root (position 0) and the
        # value is smaller than its parent: that pair breaks the heap property.
        while position > 0 and self.storage[position] < self.storage[self._parent(position)]:
            # Swap the value with its parent, moving it one level up.
            self._swap(position, self._parent(position))
            # Follow the value to its new position and check again.
            position = self._parent(position)

    def _sink_down(self) -> None:
        """Move the value at the root down until neither child is smaller
        than it, or it reaches a leaf."""
        # The value that was just moved to the top sits at position 0.
        position = 0
        # Assume the value still has work to do until we find it does not.
        sinking = True
        # A leaf has no children, so there is nowhere further down to go.
        while sinking and not self._is_leaf(position):
            # Assume the left child is the smaller one. It always exists here,
            # because this position is not a leaf.
            smaller = self._left_child(position)
            # The right child might not exist, so check that it is inside
            # the list before we look at it.
            right = self._right_child(position)
            if right < len(self.storage) and self.storage[right] < self.storage[smaller]:
                # The right child exists and is smaller, so it is the one to compare with.
                smaller = right
            if self.storage[position] > self.storage[smaller]:
                # The value is greater than its smaller child: swap them,
                # moving the value one level down.
                self._swap(position, smaller)
                # Follow the value to its new position and check again.
                position = smaller
            else:
                # Neither child is smaller: the heap property holds, so stop.
                sinking = False

    # ---- the priority queue interface

    def size(self) -> int:
        """Return the number of values in the heap."""
        # The list holds exactly the values in the heap, nothing else.
        count = len(self.storage)
        return count

    def peek(self):
        """Return the smallest value without removing it; None if empty."""
        # Pessimistic default: an empty heap has no smallest value.
        smallest = None
        if len(self.storage) > 0:
            # The heap property puts the smallest value at the root.
            smallest = self.storage[0]
        return smallest

    def add(self, value) -> None:
        """Place value in the heap, keeping the minimum heap property."""
        # Put the new value at the end: the next free spot at the bottom.
        self.storage.append(value)
        # It may be smaller than its parent, so let it float up if needed.
        self._bubble_up()

    def get(self):
        """Remove and return the smallest value; None if empty."""
        # Pessimistic default: an empty heap gives back None.
        smallest = None
        if len(self.storage) > 0:
            # The answer is the root.
            smallest = self.storage[0]
            # Remove the last value from the list so there are no gaps.
            last = self.storage.pop()
            # If that was the only value, the heap is now empty and there is
            # nothing to move; otherwise the last value takes the vacant root.
            if len(self.storage) > 0:
                self.storage[0] = last
                # It is probably too big for the top, so let it sink if needed.
                self._sink_down()
        return smallest

    # ---- printing

    def __len__(self) -> int:
        # Lets len(heap) work: it is the same as heap.size().
        return self.size()

    def __str__(self) -> str:
        # Shows the list, which is the whole tree level by level.
        return f"MinHeap363{self.storage}"

### Checking the heap

You do not need to change the heap, but run its tests so that you trust it,
and read a few of them: they show how the heap behaves.
`test_heap_property_holds` checks the list itself after every operation:
every parent must be less than or equal to both its children.

In [ ]:
# Given -- tests for the heap.
import inspect
import random


def heap_property_ok(heap: MinHeap363) -> bool:
    """True if every parent in heap.storage is <= both of its children."""
    a = heap.storage
    ok = True
    for child in range(1, len(a)):
        if a[(child - 1) // 2] > a[child]:
            ok = False
    return ok


def test_heap_empty():
    h = MinHeap363()
    assert h.size() == 0, "a new heap has size 0"
    assert h.peek() is None, "peek on an empty heap returns None"
    assert h.get() is None, "get on an empty heap returns None"
    assert h.size() == 0, "a failed get changes nothing"


def test_heap_is_leaf():
    h = MinHeap363()
    h.storage = [1, 2, 3, 4]          # set directly: tests _is_leaf alone
    assert not h._is_leaf(0), "the root has two children"
    assert not h._is_leaf(1), "position 1 has a left child at 3"
    assert h._is_leaf(2), "position 2 would have children at 5 and 6"
    assert h._is_leaf(3), "position 3 is the last leaf"


def test_heap_single_value():
    h = MinHeap363()
    h.add(42)
    assert h.size() == 1
    assert h.peek() == 42, "peek returns the only value"
    assert h.size() == 1, "peek does not remove"
    assert h.get() == 42, "get returns the only value"
    assert h.size() == 0, "and the heap is empty again"
    assert h.get() is None


def test_heap_bubble_up_class_example():
    # From class: 2, 8, 4 is a minimum heap; adding 1 floats to the root.
    h = MinHeap363()
    for v in [2, 8, 4]:
        h.add(v)
    h.add(1)
    assert h.storage == [1, 2, 4, 8], f"expected [1, 2, 4, 8], got {h.storage}"


def test_heap_sink_down_swaps_with_smaller_child():
    # 3 moves to the root and must swap with 1 (the smaller child), not 2.
    h = MinHeap363()
    h.storage = [0, 2, 1, 3]          # a valid minimum heap
    assert h.get() == 0
    assert h.storage == [1, 2, 3], f"expected [1, 2, 3], got {h.storage}"


def test_heap_ascending_and_descending_adds():
    for values in ([1, 2, 3, 4, 5, 6, 7], [7, 6, 5, 4, 3, 2, 1]):
        h = MinHeap363()
        for v in values:
            h.add(v)
        out = [h.get() for _ in values]
        assert out == [1, 2, 3, 4, 5, 6, 7], f"added {values}, got {out}"


def test_heap_duplicates():
    h = MinHeap363()
    for v in [2, 1, 2, 1, 2]:
        h.add(v)
    out = [h.get() for _ in range(5)]
    assert out == [1, 1, 2, 2, 2], f"got {out}"


def test_heap_property_holds():
    random.seed(363)
    h = MinHeap363()
    for _ in range(300):
        if random.random() < 0.6:
            h.add(random.randint(-50, 50))
        else:
            h.get()
        assert heap_property_ok(h), f"heap property broken: {h.storage}"


def test_heap_random_matches_sorted():
    random.seed(9)
    for n in [2, 3, 10, 31, 32, 33, 200]:
        values = [random.randint(0, 100) for _ in range(n)]
        h = MinHeap363()
        for v in values:
            h.add(v)
        assert h.size() == n
        out = [h.get() for _ in range(n)]
        assert out == sorted(values), f"n={n}: values came out out of order"
        assert h.get() is None


def test_heap_tuples():
    # The simulation stores (severity, arrival_minute) pairs. Python compares tuples
    # element by element, so a smaller first element wins and ties go to the second.
    h = MinHeap363()
    for patient in [(3, 1), (2, 2), (1, 3), (2, 4), (1, 5)]:
        h.add(patient)
    out = [h.get() for _ in range(5)]
    assert out == [(1, 3), (1, 5), (2, 2), (2, 4), (3, 1)], f"got {out}"


run_tests("test_heap")

---

## The emergency department

### The model

An emergency department is a priority queue with people in it. Patients are not
seen in the order they arrive; they are seen in the order of how badly they
need to be seen. Every patient has a **severity**. A smaller number is more
severe, which is exactly what a **minimum** heap returns first:

| Severity | Meaning | Who comes out of the heap first |
|---|---|---|
| `1` | life-threatening | always ahead of severities 2 and 3 |
| `2` | severe | ahead of severity 3 |
| `3` | non-life-threatening | last. You may wait 12 hours with a fever. |

Among patients with the **same** severity, the one who **arrived first** is
seen first. So the waiting room holds pairs:

```python
(severity, arrival_minute)
```

Python compares tuples element by element: a smaller severity wins, and when
two severities tie, the smaller (earlier) `arrival_minute` wins. No extra
priority number is needed. `test_heap_tuples` above checks the heap with pairs
of this shape. The arrival minute has a second use: a patient's wait is
`minute - arrival_minute`.

### The clock

A discrete event simulation follows a system through the moments when
something happens: a doctor becomes free, a patient arrives. The simplest way
to run one is to advance a clock **one minute at a time**. The simulation is
one master loop, and each minute, in this order:

1. **Treatment.** Every doctor who is free this minute calls in the next
   patient with `get`, if anyone is waiting. That patient waited
   `minute - arrival_minute` minutes; record it under their severity. The
   doctor is now busy for a random number of minutes that depends on the
   severity.
2. **Arrival.** With probability `arrival_prob`, one new patient arrives. Give
   them a random severity. If the waiting room (your `MinHeap363`) already
   holds `WAITING_ROOM_CAPACITY` patients, the patient is **turned away**: they
   are counted, but not added and never seen. Otherwise `add` the pair
   `(severity, minute)`.
3. **Statistics.** After the doctors and the arrival, update the running
   statistics for this minute: the size of the waiting room, how many doctors
   are busy, and whether the room just filled for the first time.

A doctor who starts a treatment at minute `m` that lasts `t` minutes is free
again at minute `m + t`. One list, with one entry per doctor holding *the
minute this doctor is next free*, is all the bookkeeping the doctors need.

Because treatment comes before arrival in each minute, a patient who arrives
in minute `m` can be seen at minute `m + 1` at the earliest. The shortest
possible wait is one minute.

When the clock runs out, some patients may still be in the waiting room.
They were never seen, so they have no wait time to average. **Count them
anyway.** A severity with a short average wait and 400 people still in the
waiting room is not being served well.

### The parameters

The values below are **placeholders** that behave sensibly. They are biased
toward the non-life-threatening and severe patients, as real EDs are. Your code
must read them from these names; never type the numbers into your functions.
Then the simulation keeps working when we replace them.

In [ ]:
# Given -- simulation settings. PLACEHOLDER values.
LIFE_THREATENING = 1
SEVERE = 2
NON_LIFE_THREATENING = 3
TIERS = [LIFE_THREATENING, SEVERE, NON_LIFE_THREATENING]

# Share of arrivals with each severity, in the same order as TIERS. Must add up to 1.
TIER_WEIGHTS = [0.10, 0.35, 0.55]

# Treatment length in minutes for each severity: (shortest, longest), inclusive.
TREATMENT_MINUTES = {
    LIFE_THREATENING: (60, 180),
    SEVERE: (30, 90),
    NON_LIFE_THREATENING: (10, 30),
}

ARRIVAL_PROB = 0.25          # chance of one arrival in any given minute (about 15 per hour)
NUM_DOCTORS = 12
SIM_MINUTES = 7 * 24 * 60    # one week
SEED = 363                   # same seed -> same "random" week, so runs can be compared

# Most patients the waiting room can hold at once. PLACEHOLDER.
WAITING_ROOM_CAPACITY = 60

### Before you build: what is a simulation?

You have probably not written one. Everything you need is here.

**Why simulate.** We want to know how an ED behaves when we change the
staffing or the number of patients. We cannot experiment on a real ED, and
the math for a system with random arrivals, random illnesses and random
treatments is hard. So we build a small imitation, run it on the computer,
and measure it. The imitation does not have to be real. It has to capture
the few things that make waiting happen: *limited doctors, random demand,
and an order of service.*

**State, event, clock.** A simulation has three ingredients:

- **State**: everything that describes the system right now. Ours is the
  waiting room (the heap), the list of when each doctor is free, and the
  counters.
- **Events**: the things that change the state. A patient arrives; a doctor
  starts a treatment; a treatment ends.
- **Clock**: the simulated time, advancing step by step. One simulated week
  runs in a fraction of a second because the clock does not wait.

"Discrete" means the state only changes at separate moments, not
continuously. Ours advances the clock one minute at a time and checks what
happens in that minute. (More elaborate simulators jump straight from one event to
the next. The result is the same; the minute-by-minute version is easier to
get right.)

**The random seed.** A computer does not produce truly random numbers. A
*pseudo-random number generator* computes each number from the previous one,
by a fixed formula, and the first number comes from a starting value called
the **seed**. Two consequences: (1) the numbers look random, but the same
seed always produces exactly the same sequence; (2) a different seed produces
a different sequence. In Python you set the seed with `random.seed(s)`. For
us this is useful. With the same seed, the simulation replays the same week,
so you can find a bug by running the same week again, and you can compare two
staffing levels on *the same* week, so that any difference comes from the
staffing and not from different random numbers.

**One run is one sample.** A run is not the answer; it is one possible week.
A result from one seed may be due to chance. Before you believe a difference, run
several seeds and see whether it holds. Statistics (means, percentiles,
spread) are how you summarize many runs.

**Utilization and the cliff.** If patients bring in, on average, 11 minutes
of work per minute and you have 12 doctors, the doctors are busy 92% of the
time, and the waiting room is usually short, although its length varies from
minute to minute. With 10 doctors, work arrives faster than the doctors can
finish it, and the waiting room grows until it is full, and then patients are turned away. The change between those
two situations is sudden, not gradual (we call it the "cliff"). Queues behave
this way, and it is the main thing this assignment shows you.

#### A hand-trace to do first (10 minutes, on paper)

Before coding, simulate this by hand: **1 doctor**, treatments always exactly
3 minutes, and these arrivals as `(minute, severity)`: `(0, 3)`, `(1, 2)`,
`(2, 3)`, `(3, 1)`, `(4, 3)`. For minutes 0 to 10, write down for every minute
the waiting room (as `(severity, arrival_minute)` pairs), the doctor's
`free_at`, and who is called in, with what wait. Remember the order inside a
minute: treatment first, then arrival. Notice two things: the severity-1
patient who arrived at minute 3 is called ahead of patients who arrived
earlier, and the two severity-3 patients still waiting are called in the order
they arrived. Once your trace and your program agree on a tiny case, you can
trust the program on a large one.

#### How to build it, step by step

1. Read the heap and run its tests.
2. Write the three random helpers; run their tests.
3. Write `simulate` with **one doctor**; print the waiting room each minute
   for the first 20 minutes and compare with your trace.
4. Make the number of doctors a parameter. Then run the tests.
5. Add the waiting-room capacity, then the per-minute statistics, re-running the tests after each.

**Common mistakes**

- Calling `random` inside `simulate`: the draws then change when you reorder code, and results change for no visible reason.
- Using `randint(a, b)` as if `b` were excluded.
- Forgetting patients still in the heap after the loop (they must be counted).
- Forgetting the patients who were turned away: they arrived, so they must still be in the books.
- Updating the statistics in the middle of the minute instead of after both the doctors and the arrival.
- Setting a doctor's `free_at` to the treatment length instead of `minute + length`.
- Looping over doctors but calling `get` on an empty heap: check `size() > 0` each time.
- Comparing means from different seeds, or changing two parameters at once.
- Trusting an average when many patients are still waiting. They have no wait recorded, so they are missing from it.

### Reading: random numbers in Python

Everything random in this simulation comes from Python's `random` module.
Read the
[`random` documentation](https://docs.python.org/3/library/random.html),
at least the sections *Bookkeeping functions*, *Functions for integers* and
*Functions for sequences*. Then try the cell below and make sure you can
explain every line of its output.

Four functions are all you need:

- `random.seed(s)` fixes the starting point of the random number generator.
  The same seed produces the same sequence of "random" numbers every time.
  That is what makes a simulation repeatable, and it is what lets the tests
  check your work.
- `random.random()` returns a float in $[0, 1)$. "With probability $p$" in
  code is `random.random() < p`.
- `random.randint(a, b)` returns an integer from `a` to `b`, **including
  both ends**. (Most of Python excludes the right end; this function does
  not.)
- `random.choices(population, weights=...)` returns a **list** of picks.
  It picks one item by default, so you need `[0]` to get the item itself.

In [ ]:
# Given -- experiment with the random module. Change the seed and re-run.
import random

random.seed(SEED)
print("five floats:     ", [round(random.random(), 3) for _ in range(5)])
print("five dice rolls: ", [random.randint(1, 6) for _ in range(5)])
print("one weighted pick:", random.choices(TIERS, weights=TIER_WEIGHTS))

picks = random.choices(TIERS, weights=TIER_WEIGHTS, k=10_000)
print("share of each tier in 10,000 picks:",
      {tier: picks.count(tier) / len(picks) for tier in TIERS})

### What you write

Three small functions, then the simulation itself. The three small ones are
where all the randomness lives. `simulate` should not call anything from
`random` except `random.seed`.

`simulate` returns its results as a dictionary with one entry per severity,
plus one entry, `"room"`, for the waiting room and the doctors:

```python
{
    LIFE_THREATENING:     {"arrived": ..., "seen": ..., "total_wait": ...,
                           "still_waiting": ..., "turned_away": ...},
    SEVERE:               {...},
    NON_LIFE_THREATENING: {...},
    "room":               {"peak_waiting": ..., "first_full_minute": ...,
                           "waiting_sum": ..., "busy_sum": ...},
}
```

- `arrived`: how many patients of this severity arrived (including those turned away).
- `seen`: how many were called in by a doctor.
- `total_wait`: the sum of the waits of the patients who were seen.
- `still_waiting`: how many were still in the waiting room when the clock ran out.
- `turned_away`: how many were refused because the waiting room was full.

For every severity, `arrived == seen + still_waiting + turned_away`. Nobody
leaves the ED without being counted.

The `"room"` numbers are updated once per minute, after the doctors and the
arrival:

- `peak_waiting`: the largest waiting-room size seen at the end of any minute.
- `first_full_minute`: the first minute at whose end the room held
  `capacity` patients, or `None` if that never happened.
- `waiting_sum`: the sum, over all minutes, of the waiting-room size. Divide by
  the number of minutes for the average queue length.
- `busy_sum`: the sum, over all minutes, of the number of busy doctors. A doctor
  who starts a treatment at minute `m` lasting `t` minutes is busy during
  minutes `m` to `m + t - 1`. Divide by `num_doctors * minutes` for utilization.

Should a life-threatening patient be turned away because the room is full of
severity-3 patients? Our rule says yes: the room is full, whoever arrives.
Question 6 asks what you think of that.

In [ ]:
# TODO -- write the three helpers and simulate().
def random_tier() -> int:
    """Return a random severity from TIERS, chosen according to TIER_WEIGHTS."""
    tier = NON_LIFE_THREATENING
    # TODO
    return tier


def arrival_happens(arrival_prob: float) -> bool:
    """Return True with probability arrival_prob."""
    happens = False
    # TODO
    return happens


def treatment_minutes(tier: int) -> int:
    """Return a random treatment length for tier, within TREATMENT_MINUTES[tier]."""
    minutes = 0
    # TODO
    return minutes


def simulate(num_doctors: int, arrival_prob: float, minutes: int,
             capacity: int = WAITING_ROOM_CAPACITY,
             seed: int | None = None) -> dict:
    """Run the ED for `minutes` minutes and return the statistics.

    Each minute: every free doctor calls in the most urgent waiting patient
    from a MinHeap363; then at most one new patient arrives and is added to
    the heap as (severity, arrival_minute), or turned away if the heap
    already holds `capacity` patients; then the statistics are updated.
    """
    random.seed(seed)
    stats = {tier: {"arrived": 0, "seen": 0, "total_wait": 0,
                    "still_waiting": 0, "turned_away": 0}
             for tier in TIERS}
    stats["room"] = {"peak_waiting": 0, "first_full_minute": None,
                     "waiting_sum": 0, "busy_sum": 0}
    # TODO
    return stats

In [ ]:
# Given -- turning statistics into a report.
TIER_NAMES = {LIFE_THREATENING: "life-threatening",
              SEVERE: "severe",
              NON_LIFE_THREATENING: "non-life-threatening"}


def average_wait(stats: dict, tier: int) -> float | None:
    """Average wait, in minutes, of the patients of `tier` who were seen."""
    average = None
    if stats[tier]["seen"] > 0:
        average = stats[tier]["total_wait"] / stats[tier]["seen"]
    return average


def print_report(stats: dict) -> None:
    print(f"{'severity':22s} {'arrived':>8s} {'seen':>6s} {'waiting':>8s} {'turned away':>12s} {'avg wait (min)':>15s}")
    for tier in TIERS:
        s = stats[tier]
        avg = average_wait(stats, tier)
        shown = "--" if avg is None else f"{avg:.1f}"
        print(f"{TIER_NAMES[tier]:22s} {s['arrived']:8d} {s['seen']:6d} "
              f"{s['still_waiting']:8d} {s['turned_away']:12d} {shown:>15s}")
    room = stats["room"]
    print(f"waiting room: peak {room['peak_waiting']}, first full at minute {room['first_full_minute']}")


print_report(simulate(NUM_DOCTORS, ARRIVAL_PROB, SIM_MINUTES, seed=SEED))

### Tests for the simulation

Randomness makes exact answers impossible to test, so these tests check
things that must be true no matter what the dice say: counts add up (including
the patients turned away), arrivals stop when `arrival_prob` is 0, a tiny
waiting room turns patients away and never holds more than its capacity,
the statistics stay within what is possible, with plenty of doctors everyone waits
exactly one minute, the most severe patients wait less than the least severe
ones when doctors are scarce, and the same seed gives the same week.

In [ ]:
# Given -- tests for the simulation. Run this cell after every change.
def test_ed_random_tier_follows_weights():
    random.seed(1)
    picks = [random_tier() for _ in range(20_000)]
    for tier, weight in zip(TIERS, TIER_WEIGHTS):
        share = picks.count(tier) / len(picks)
        assert abs(share - weight) < 0.02, f"tier {tier}: share {share:.3f}, expected about {weight}"


def test_ed_arrival_happens_extremes_and_rate():
    random.seed(2)
    assert not any(arrival_happens(0.0) for _ in range(1000)), "probability 0 never happens"
    assert all(arrival_happens(1.0) for _ in range(1000)), "probability 1 always happens"
    share = sum(arrival_happens(0.25) for _ in range(20_000)) / 20_000
    assert abs(share - 0.25) < 0.02, f"probability 0.25 happened {share:.3f} of the time"


def test_ed_treatment_minutes_in_range():
    random.seed(3)
    for tier in TIERS:
        low, high = TREATMENT_MINUTES[tier]
        seen = {treatment_minutes(tier) for _ in range(5000)}
        assert min(seen) >= low and max(seen) <= high, f"tier {tier}: out of [{low}, {high}]"
        assert low in seen and high in seen, f"tier {tier}: both ends must be possible"


def test_ed_nobody_arrives():
    stats = simulate(num_doctors=3, arrival_prob=0.0, minutes=600, seed=4)
    for tier in TIERS:
        assert stats[tier] == {"arrived": 0, "seen": 0, "total_wait": 0, "still_waiting": 0, "turned_away": 0}
    room = stats["room"]
    assert room["peak_waiting"] == 0 and room["first_full_minute"] is None
    assert room["waiting_sum"] == 0 and room["busy_sum"] == 0


def test_ed_everyone_is_counted():
    stats = simulate(num_doctors=4, arrival_prob=0.3, minutes=2000, seed=5)
    assert sum(stats[t]["arrived"] for t in TIERS) > 400, "about 600 arrivals expected"
    for tier in TIERS:
        s = stats[tier]
        assert s["arrived"] == s["seen"] + s["still_waiting"] + s["turned_away"], f"tier {tier}: {s}"


def test_ed_plenty_of_doctors_means_one_minute_wait():
    # More doctors than patients: treatment comes before arrival in each
    # minute, so everyone is called in the minute after they arrive.
    stats = simulate(num_doctors=1000, arrival_prob=0.5, minutes=500, seed=6)
    for tier in TIERS:
        s = stats[tier]
        assert s["arrived"] > 0
        assert s["total_wait"] == s["seen"], f"tier {tier}: everyone should wait exactly 1 minute: {s}"
    assert sum(stats[t]["still_waiting"] for t in TIERS) <= 1, "only the last minute's arrival can still be waiting"


def test_ed_priority_matters_when_busy():
    # A busy week, but with enough doctors that every tier gets seen.
    stats = simulate(num_doctors=11, arrival_prob=0.25, minutes=SIM_MINUTES, seed=7)
    waits = [average_wait(stats, tier) for tier in TIERS]
    assert None not in waits, f"every tier should have patients seen: {waits}"
    assert waits[0] < 30, f"life-threatening waited {waits[0]:.0f} min on average"
    assert waits[0] < waits[1] < waits[2], f"waits should grow with the tier number: {waits}"


def test_ed_small_waiting_room_turns_patients_away():
    stats = simulate(num_doctors=1, arrival_prob=0.5, minutes=1000, capacity=5, seed=9)
    room = stats["room"]
    assert sum(stats[t]["turned_away"] for t in TIERS) > 0, "a 5-seat room with 1 doctor must turn patients away"
    assert room["peak_waiting"] == 5, f"peak {room['peak_waiting']}, expected exactly the capacity, 5"
    assert room["first_full_minute"] is not None
    for tier in TIERS:
        s = stats[tier]
        assert s["arrived"] == s["seen"] + s["still_waiting"] + s["turned_away"], f"tier {tier}: {s}"


def test_ed_big_waiting_room_never_fills():
    stats = simulate(num_doctors=1000, arrival_prob=0.5, minutes=500, capacity=50, seed=6)
    assert all(stats[t]["turned_away"] == 0 for t in TIERS)
    assert stats["room"]["first_full_minute"] is None
    assert stats["room"]["peak_waiting"] <= 1, "with 1000 doctors at most one patient waits, for one minute"


def test_ed_statistics_are_possible():
    doctors, minutes = 4, 2000
    stats = simulate(num_doctors=doctors, arrival_prob=0.3, minutes=minutes, capacity=20, seed=5)
    room = stats["room"]
    assert 0 < room["busy_sum"] <= doctors * minutes, f"busy_sum {room['busy_sum']}"
    assert 0 < room["waiting_sum"] <= 20 * minutes, f"waiting_sum {room['waiting_sum']}"
    assert 0 < room["peak_waiting"] <= 20


def test_ed_overloaded_doctor_is_always_busy():
    # One doctor and far more work than one doctor can do: busy almost every minute.
    stats = simulate(num_doctors=1, arrival_prob=0.9, minutes=3000, capacity=10, seed=3)
    assert stats["room"]["busy_sum"] >= 0.95 * 3000, f"busy_sum {stats['room']['busy_sum']} of 3000"


def test_ed_same_seed_same_week():
    a = simulate(num_doctors=10, arrival_prob=0.25, minutes=3000, seed=8)
    b = simulate(num_doctors=10, arrival_prob=0.25, minutes=3000, seed=8)
    assert a == b, "the same seed must reproduce the same results"


def test_ed_uses_the_heap():
    src = inspect.getsource(simulate)
    assert "MinHeap363()" in src, "the waiting room must be a MinHeap363"
    for banned in ["heapq", "sorted(", ".sort("]:
        assert banned not in src, f"simulate may not use {banned}"


run_tests("test_ed")

### Experiments

Once the simulation tests pass, run the cell below. It keeps the week, the seed
and the arrival rate fixed, and changes only the number of doctors.

In [ ]:
# Given -- how many doctors does this ED need?
print(f"arrival_prob = {ARRIVAL_PROB}, one week, seed = {SEED}\n")
print(f"{'doctors':>7s}  " + "  ".join(f"{TIER_NAMES[t]:>22s}" for t in TIERS))
for doctors in range(6, 17):
    stats = simulate(doctors, ARRIVAL_PROB, SIM_MINUTES, seed=SEED)
    cells = []
    for tier in TIERS:
        avg = average_wait(stats, tier)
        shown = "--" if avg is None else f"{avg:.0f}"
        cells.append(f"{shown:>10s} min, {stats[tier]['still_waiting']:4d} left, {stats[tier]['turned_away']:4d} out")
    print(f"{doctors:7d}  " + "  ".join(f"{c:>22s}" for c in cells))

Now add your own experiment cells below this one. Change one thing at a
time and keep the seed fixed, so that the only difference between two runs
is the thing you changed. Then answer these questions in your report, in
your own words:

1. **The cliff.** In the table above, the waits do not grow smoothly as
   doctors are removed. Somewhere they jump. Where is the jump? Using
   `ARRIVAL_PROB`, `TIER_WEIGHTS` and `TREATMENT_MINUTES`, estimate how many
   minutes of treatment arrive per minute of clock time on average. How does
   that number explain where the jump is?
2. **Who pays for a short staff?** When there are too few doctors, which
   severity's wait grows first and fastest? Which barely notices? Why does a
   priority queue produce exactly this pattern? Then compare, in
   the row for the fewest doctors, the average wait of the patients who were
   seen with the number turned away and still waiting. What does the average
   leave out?
3. **Half the staff calls in sick.** Start from a staffing level that keeps
   every severity's average wait under an hour, then halve it. Describe what
   happens to each severity.
4. **A busier night.** Keep `NUM_DOCTORS` fixed and raise `ARRIVAL_PROB`.
   How much busier can the ED get before the non-life-threatening patients
   are effectively never seen within the week?
5. **What the heap does.** Over a simulated week, about how many times is
   `add` called, and how many times is `get` called? If the waiting room
   were an unsorted list scanned for its most urgent patient instead, what
   would each `get` cost, and why does it matter more on a bad day than on a
   quiet one?
6. **Turning patients away.** Our rule turns away whoever arrives when the room is
   full, whatever their severity. Is that acceptable? Look at the `turned_away`
   counts by severity in a bad week. What would you change, and what would it cost?

### Optional extensions

- **Not every hour is the same.** Make `ARRIVAL_PROB` change with the hour:
  busier in the evening, quieter at 4 a.m.
- **Make room for the worst cases.** Let a life-threatening patient who arrives at
  a full room take the place of the least urgent patient waiting.
- **Surviving the wait.** Give each severity a maximum safe wait. Count the
  patients who are seen after it, or never seen.

## What to submit

Turn in **Python files and one PDF**. Not a notebook.

- **`heap363.py`**: the given `MinHeap363` class, unchanged.
- **`ed_simulation.py`**: the settings, the three random helpers, `simulate`,
  and `average_wait` / `print_report`. It must `from heap363 import MinHeap363`,
  and it must run from a terminal with `python ed_simulation.py` and print the
  report without errors.
- **`test_heap363.py`** and **`test_ed_simulation.py`**: the Given tests,
  unchanged. They must pass; the output of running them is part of your grade.
- **`report.pdf`**: your answers to the six questions, with any table or graph
  that supports a claim and a caption saying what the reader should see.
  Explain the seed in your own words: what it is, why the same seed reproduces
  the same week, and what a result from a single seed can and cannot tell you.
  One to two pages.